# Inference Acceleration: Why Is Your LLM So Slow?

> **Background**: you ask an LLM a question and it streams tokens one by one. Why can't it output everything at once? Why is it sometimes fast and sometimes slow?
>
> Goal for this part: **understand the real root causes of slow inference, and how KV Cache, FlashAttention, and vLLM speed things up.**

There is one core issue:
**during autoregressive decoding, every new token would naively re-compute attention over all previous tokens.**
It is like re-reading the whole essay from the beginning after writing each new word.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)

### 1. The root cause: repeated computation

Autoregressive generation:

```
Step 1: input [BOS]                    -> attention -> predict "I"
Step 2: input [BOS, I]                 -> attention -> predict "love"
Step 3: input [BOS, I, love]           -> attention -> predict "you"
Step 4: input [BOS, I, love, you]      -> attention -> predict EOS
```

Problem: in Step 2 you re-compute K and V for `[BOS]` again.
In Step 3 you re-compute everything from Step 1+2 again.

```
Step 1: compute K1,V1 (1 token)
Step 2: compute K1,V1,K2,V2 (2 tokens; K1,V1 are repeated)
Step 3: compute K1,V1,K2,V2,K3,V3 (3 tokens; first 4 are repeated)
...
```

Generating N tokens costs about `1+2+...+N = O(N^2)` attention work.
For N=100, that's 5050 "token-attention" computations, not 100.


In [ ]:
# 可视化repeated computation
def naive_generation_cost(n_tokens):
    """朴素自回归生成的总计算量（以 token 为单位）"""
    return sum(range(1, n_tokens + 1))

for n in [10, 50, 100, 500]:
    cost = naive_generation_cost(n)
    print(f"生成 {n:3d} 个 token -> 实际算了 {cost:6d} 次 token attention -> 浪费 {cost - n:5d} 次")

print(f"\n生成 100 个 token，浪费了 {naive_generation_cost(100) - 100} 次计算！")
print(f"这就是 LLM inferenceslow的根本原因。")

### 2. Solution 1: KV Cache (store what you already computed)

Core idea: K and V for previous tokens are already computed. Store them and reuse them.

```
Step 1: compute K1,V1 -> store in cache
Step 2: compute only K2,V2 -> read K1,V1 from cache -> concatenate
Step 3: compute only K3,V3 -> read previous K,V from cache -> concatenate
...
```

Effect: total compute drops from `O(N^2)` to `O(N)`.

```
Naive:    1+2+...+100 = 5050
KV cache: 1+1+...+1   = 100   (about 50x fewer)
```


In [ ]:
class AttentionWithKVCache(nn.Module):
    """
    带 KV Cache 的 Self-Attention
    
    Key区别：
    - 第一次调用（prefill）：处理整个 prompt，算所有 K、V 并存起来
    - 后续调用（decode）：只算新 token 的 K、V，从 cache 取旧的
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        # KV Cache 存储
        self.k_cache = None
        self.v_cache = None
    
    def reset_cache(self):
        """清空 cache（开始新的生成时调用）"""
        self.k_cache = None
        self.v_cache = None
    
    def forward(self, x, use_cache=True):
        """
        输入: x shape = [batch, seq_len, d_model]
        
        如果 use_cache=True 且 cache 非空：
            x 只包含新 token（seq_len=1）
            从 cache 取旧的 K、V，只算新的 K、V
        """
        batch_size, seq_len, _ = x.shape
        
        # 算 Q、K、V
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        if use_cache:
            if self.k_cache is not None:
                # 拼接旧的 K、V
                K = torch.cat([self.k_cache, K], dim=2)
                V = torch.cat([self.v_cache, V], dim=2)
            # 更新 cache
            self.k_cache = K
            self.v_cache = V
        
        # Attention 计算（和之前一样）
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        attn_output = weights @ V
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)

print("带 KV Cache 的 Attention 定义完成！")

In [ ]:
# 对比：有 KV Cache vs 没有
attn = AttentionWithKVCache(d_model=64, num_heads=4)

# === 模拟生成 20 个 token ===
N = 20

# 无 cache 版本（朴素）
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=False)
    sequence = torch.cat([sequence, torch.randn(1, 1, 64)], dim=1)
no_cache_time = time.time() - start

# 有 cache 版本
attn.reset_cache()
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=True)
    sequence = torch.randn(1, 1, 64)  # 每次只输入新 token
cache_time = time.time() - start

print(f"无 KV Cache: {no_cache_time*1000:.1f} ms")
print(f"有 KV Cache: {cache_time*1000:.1f} ms")
print(f"acceleration比: {no_cache_time/cache_time:.1f}x")
print(f"\nNote：N=20 时差别不大，但 N=1000 时差别是巨大的！")

### 3. The memory cost of KV Cache

KV cache fixes compute, but introduces a **memory problem**.

A rough estimate (FP16):

```
KV cache per token ~= 2 * num_layers * d_model * 2 bytes

LLaMA-7B: 32 layers * 4096 dims * 2 * 2 bytes  ~= 1 MB / token
  2048 tokens -> ~2 GB KV cache

LLaMA-70B: 80 layers * 8192 dims * 2 * 2 bytes ~= 5 MB / token
  2048 tokens -> ~10 GB KV cache
```

And each request needs its own KV cache. 100 concurrent users * 2GB = 200GB VRAM.

This is a core problem vLLM is designed to address.


### 4. vLLM's key idea: PagedAttention

Problem: KV cache is usually allocated as one big contiguous chunk per request. But different requests produce different lengths, which leads to **fragmentation**.
Like a parking lot with cars of different sizes leaving unusable gaps.

vLLM's idea: split KV cache into fixed-size "pages" (blocks), managed like virtual memory.

```
Traditional (contiguous):
+------------------------------+
| request A KV (500 tokens)    |
+------------------------------+
| wasted fragment               |
+--------------+
| req B (100)  |
+--------------+

PagedAttention (paged):
+--+--+--+--+--+--+--+--+
|A1|B1|A2|C1|B2|A3|C2|A4|  each page fixed size (e.g. 16 tokens)
+--+--+--+--+--+--+--+--+
non-contiguous but far less fragmentation
```

Effect:
- memory utilization improves from ~20-40% to near 100%
- same VRAM can serve 2-4x more concurrent requests


In [ ]:
# 直观感受：连续分配 vs 分页的内存浪费
import random

def simulate_memory(requests, block_size=16, total_memory=1024):
    """模拟两种内存分配策略"""
    
    # 连续分配
    contiguous_used = 0
    contiguous_wasted = 0
    for req_len in requests:
        contiguous_used += req_len
        # 对齐到 64 的倍数（实际中更复杂）
        aligned = ((req_len + 63) // 64) * 64
        contiguous_wasted += (aligned - req_len)
    
    # 分页分配
    paged_used = 0
    paged_wasted = 0
    for req_len in requests:
        paged_used += req_len
        # 只浪费最后一页没填满的部分
        paged_wasted += (block_size - (req_len % block_size)) % block_size
    
    return contiguous_wasted, paged_wasted

# 模拟 50 个随机长度的请求
random.seed(42)
requests = [random.randint(10, 500) for _ in range(50)]

cw, pw = simulate_memory(requests)
print(f"50 个请求，总 token 数: {sum(requests)}")
print(f"连续分配浪费: {cw} 个 token 位置 ({cw/sum(requests)*100:.1f}%)")
print(f"分页分配浪费: {pw} 个 token 位置 ({pw/sum(requests)*100:.1f}%)")
print(f"\n分页节省了 {cw-pw} 个 token 位置的内存！")

### 5. FlashAttention: make attention compute faster

KV cache reduces *how much* you compute.
FlashAttention reduces *how slow* attention is, by fixing memory access.

Problem: the bottleneck is often not FLOPs, but **HBM (VRAM) bandwidth**.

Standard attention does multiple read/write passes to VRAM:
1. QK^T -> write to HBM
2. softmax -> read, compute, write
3. multiply by V -> read, compute, write

FlashAttention idea:
- tile Q,K,V into blocks
- stream blocks from HBM into on-chip SRAM
- compute attention fully in SRAM
- write only the final output back to HBM
- do NOT write intermediate matrices to HBM

Effect:
- 2-4x speedup
- 10-20x less VRAM usage for intermediates
- mathematically exact, not an approximation


In [ ]:
# 直观感受：VRAM读写 vs 计算的时间
# 这些数字是近似值，用于建立直觉

print("=== GPU 各级存储的速度对比 ===")
print()
print("HBM (VRAM):    1.5 TB/s 带宽，但离计算单元远")
print("SRAM (片上):   19 TB/s 带宽，紧挨计算单元")
print()
print("标准 Attention 读写 HBM 的量:")
print("  Q×K^T 矩阵: seq_len² × 2 bytes")
print("  对于 2048 token: 2048² × 2 = 8MB")
print("  对于 8192 token: 8192² × 2 = 128MB ← 巨大！")
print()
print("FlashAttention: 这个矩阵不写回 HBM，省了 128MB 的读写")
print("这就是 FlashAttention 快的原因。")

### 6. Two phases of inference: Prefill vs Decode

LLM inference has two phases with very different bottlenecks:

| | Prefill | Decode |
|:---|:---|:---|
| What | process the full prompt once | generate new tokens one by one |
| Input length | long (thousands) | short (1 token per step) |
| Bottleneck | **compute** | **memory bandwidth** (reading KV cache) |
| Main optimizations | FlashAttention | KV cache + PagedAttention |
| Parallelism | high (tokens parallel) | low (serial) |

This is why the first token is often slow (prefill), and later tokens are faster (decode + KV cache).


### 7. Other acceleration techniques (at a glance)

| Technique | What | Speedup | Accuracy loss |
|:---|:---|:---|:---|
| **KV Cache** | reuse past K,V | 10-50x | none |
| **FlashAttention** | reduce HBM traffic | 2-4x | none |
| **PagedAttention (vLLM)** | paged KV management | 2-4x throughput | none |
| **Quantization (INT8/INT4)** | fewer bits for weights | 2-4x | small |
| **Speculative decoding** | draft+verify | 2-3x | none |
| **Continuous batching** | admit requests continuously | 5-10x throughput | none |
| **Tensor parallelism** | split layers across GPUs | ~#GPUs | none |

Speculative decoding is especially interesting; we cover it in the next part.


---

## Inference Acceleration Summary

1. [ok] Root cause: repeated K,V computation during autoregressive decoding -> O(N^2)
2. [ok] **KV cache**: store K,V -> O(N)
3. [ok] Memory issue: long sequences + concurrency -> KV cache explodes
4. [ok] **vLLM / PagedAttention**: paged KV cache -> reduce fragmentation -> 2-4x throughput
5. [ok] **FlashAttention**: compute in SRAM, avoid intermediate HBM writes -> 2-4x faster
6. [ok] Prefill (compute-bound) vs Decode (bandwidth-bound)
7. [ok] First token is slow (prefill), later tokens are faster (decode + KV cache)

One-sentence summary: slow inference is repeated compute -> KV cache reuses -> memory becomes the next limit -> vLLM pages it -> FlashAttention makes each attention step faster.
